In [ ]:
!pip -q install imbalanced-learn xgboost lightgbm

In [ ]:
#Importing the libraries necessary
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_score, recall_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [ ]:
from google.colab import files
files.upload()



In [ ]:
# =========================================================
# LIGHTGBM + STRATIFIED 5-FOLD + INTERACTIONS
# =========================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

# -----------------------------
# LOAD
# -----------------------------
train_df = pd.read_excel("Train.xlsx")
test_df  = pd.read_excel("Test.xlsx")

TARGET = "Diabetes_binary"

ID_COL = next((c for c in ["ID","id","Id"] if c in test_df.columns), None)
test_ids = test_df[ID_COL].values if ID_COL else np.arange(len(test_df))

# -----------------------------
# FEATURES
# -----------------------------
drop_cols = [TARGET] + [c for c in ["ID","id","Id"] if c in train_df.columns]
features = [c for c in train_df.columns if c not in drop_cols]

X = train_df[features].copy()
y = train_df[TARGET].astype(int).copy()
X_test = test_df[features].copy()

# -----------------------------
# SAFE INTERACTION FEATURES (THE EDGE)
# -----------------------------
def add_interactions(df):
    if {"BMI","Age"}.issubset(df.columns):
        df["BMI_x_Age"] = df["BMI"] * df["Age"]
    if {"BMI","HighBP"}.issubset(df.columns):
        df["BMI_x_HighBP"] = df["BMI"] * df["HighBP"]
    if {"BMI","PhysActivity"}.issubset(df.columns):
        df["BMI_x_PhysActivity"] = df["BMI"] * df["PhysActivity"]
    if {"GenHlth","Age"}.issubset(df.columns):
        df["GenHlth_x_Age"] = df["GenHlth"] * df["Age"]
    if {"PhysHlth","Age"}.issubset(df.columns):
        df["PhysHlth_x_Age"] = df["PhysHlth"] * df["Age"]
    if {"MentHlth","Age"}.issubset(df.columns):
        df["MentHlth_x_Age"] = df["MentHlth"] * df["Age"]
    return df

X = add_interactions(X)
X_test = add_interactions(X_test)

# -----------------------------
# MISSING VALUES
# -----------------------------
for col in X.columns:
    med = X[col].median()
    X[col] = X[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

# -----------------------------
# CATEGORICAL FEATURES
# -----------------------------
for col in X.columns:
    if pd.api.types.is_integer_dtype(X[col]) and X[col].nunique() <= 12:
        X[col] = X[col].astype("category")
        X_test[col] = X_test[col].astype("category")

# -----------------------------
# CLASS IMBALANCE
# -----------------------------
neg, pos = (y==0).sum(), (y==1).sum()
spw = neg / pos

# -----------------------------
# MODEL PARAMS (RANK-FOCUSED)
# -----------------------------
params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_child_samples": 30,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.3,
    "reg_lambda": 1.2,
    "scale_pos_weight": spw,
    "n_jobs": -1,
    "verbose": -1,
    "random_state": 42
}

# -----------------------------
# STRATIFIED 5-FOLD CV
# -----------------------------
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

print("\n🔥 STRATIFIED 5-FOLD CV STARTING...")

for fold, (tr, va) in enumerate(kf.split(X, y), 1):
    model = lgb.LGBMClassifier(**params, n_estimators=7000)

    model.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(250, verbose=False)]
    )

    oof[va] = model.predict_proba(X.iloc[va])[:,1]
    test_pred += model.predict_proba(X_test)[:,1] / kf.n_splits

    print(f"Fold {fold} AUC:", roc_auc_score(y.iloc[va], oof[va]))

base_auc = roc_auc_score(y, oof)
print("\n⚔️ BASE OOF AUC:", round(base_auc,5))

# -----------------------------
# CALIBRATION (FINAL POLISH)
# -----------------------------
cal = LogisticRegression(max_iter=1000)
cal.fit(oof.reshape(-1,1), y)

oof_cal = cal.predict_proba(oof.reshape(-1,1))[:,1]
final_auc = roc_auc_score(y, oof_cal)

print("🏆 CALIBRATED OOF AUC:", round(final_auc,5))

test_cal = cal.predict_proba(test_pred.reshape(-1,1))[:,1]

# -----------------------------
# SUBMISSION
# -----------------------------
submission = pd.DataFrame({
    ID_COL if ID_COL else "ID": test_ids,
    TARGET: test_cal
})

submission.to_csv("submission_AG3.csv", index=False)